# Complete Implementation: Hybrid Transformer-Variational Quantum Classifier for NIDS

This notebook is a comprehensive implementation of a **Hybrid Transformer-Variational Quantum Classifier (VQC)** for Network Intrusion Detection Systems (NIDS) using the **NSL-KDD** dataset.


In [1]:
# 1. Environment Setup & Dependencies
!pip install pennylane qiskit torch pandas numpy scikit-learn matplotlib seaborn requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 70.8 MB/s eta 0:00:00


In [2]:
import os
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pennylane as qml

import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

## 2. Data Acquisition (NSL-KDD Dataset)

In [3]:
train_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt"
test_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"
train_file = "KDDTrain+.txt"
test_file = "KDDTest+.txt"
if not os.path.exists(train_file):
    urllib.request.urlretrieve(train_url, train_file)
if not os.path.exists(test_file):
    urllib.request.urlretrieve(test_url, test_file)
print("Datasets are ready!")

Datasets are ready!


In [4]:
columns = (['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
            'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
            'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count',
            'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
            'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
            'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
            'dst_host_srv_rerror_rate', 'attack_type', 'level'])
df_train = pd.read_csv(train_file, names=columns)
df_test = pd.read_csv(test_file, names=columns)
print(f"Train shape: {df_train.shape}")

Train shape: (125973, 43)


## 3. Exploratory Data Analysis & Preprocessing

In [5]:
df_train['label'] = df_train['attack_type'].apply(lambda x: 0 if x == 'normal' else 1)
df_test['label'] = df_test['attack_type'].apply(lambda x: 0 if x == 'normal' else 1)
df_train.drop(['attack_type', 'level'], axis=1, inplace=True)
df_test.drop(['attack_type', 'level'], axis=1, inplace=True)

categorical_cols = ['protocol_type', 'service', 'flag']
combined = pd.concat([df_train, df_test], axis=0)
for col in categorical_cols:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col])

df_train_proc = combined.iloc[:len(df_train), :]
df_test_proc = combined.iloc[len(df_train):, :]

X_train = df_train_proc.drop('label', axis=1).values
y_train = df_train_proc['label'].values
X_test = df_test_proc.drop('label', axis=1).values
y_test = df_test_proc['label'].values

scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def create_sequences(data, labels, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length + 1):
        xs.append(data[i:(i + seq_length)])
        ys.append(labels[i + seq_length - 1])
    return np.array(xs), np.array(ys)

seq_length = 5
X_seq_train, y_seq_train = create_sequences(X_train_scaled, y_train, seq_length)
X_seq_test, y_seq_test = create_sequences(X_test_scaled, y_test, seq_length)

X_train_sub, _, y_train_sub, _ = train_test_split(X_seq_train, y_seq_train, train_size=2000, random_state=42, stratify=y_seq_train)
X_test_sub, _, y_test_sub, _ = train_test_split(X_seq_test, y_seq_test, train_size=500, random_state=42, stratify=y_seq_test)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train_sub, dtype=torch.float32), torch.tensor(y_train_sub, dtype=torch.long)), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test_sub, dtype=torch.float32), torch.tensor(y_test_sub, dtype=torch.long)), batch_size=32, shuffle=False)
input_dim = X_train_sub.shape[2]

## 4. Improved Classical Transformer Encoder
Includes Positional Encoding and non-linear dimensionality reduction to prevent representational collapse.

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerEncoder(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2, n_qubits=4):
        super(TransformerEncoder, self).__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, dropout=0.1)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc_reduce = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Linear(16, n_qubits)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        x = self.fc_reduce(x)
        x = torch.tanh(x) * np.pi
        return x

## 5. Variational Quantum Classifier (VQC)

In [7]:
n_qubits = 4
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation='Y')
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (n_layers, n_qubits, 3)}
init_method = {"weights": lambda x: torch.nn.init.uniform_(x, 0, 2 * np.pi)}
qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes, init_method=init_method)

## 6. Hybrid Architecture & Training

In [8]:
class HybridTransformerVQC(nn.Module):
    def __init__(self, input_dim, n_qubits=4, num_classes=2):
        super(HybridTransformerVQC, self).__init__()
        self.transformer = TransformerEncoder(input_dim, d_model=64, nhead=4, num_layers=2, n_qubits=n_qubits)
        self.vqc = qlayer
        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 4),
            nn.ReLU(),
            nn.Linear(4, num_classes)
        )

    def forward(self, x):
        x = self.transformer(x)
        x = self.vqc(x)
        x = self.classifier(x)
        return x

model = HybridTransformerVQC(input_dim=input_dim, n_qubits=n_qubits)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
epochs = 20

print("Training Started...")
train_losses = []
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")


Training Started...
Epoch 1/20 - Loss: 0.6780
Epoch 2/20 - Loss: 0.6085
Epoch 3/20 - Loss: 0.5988
Epoch 4/20 - Loss: 0.4653
Epoch 5/20 - Loss: 0.3450
Epoch 6/20 - Loss: 0.2769
Epoch 7/20 - Loss: 0.2639
Epoch 8/20 - Loss: 0.2524
Epoch 9/20 - Loss: 0.2449
Epoch 10/20 - Loss: 0.2613
Epoch 11/20 - Loss: 0.2553
Epoch 12/20 - Loss: 0.2769
Epoch 13/20 - Loss: 0.2406
Epoch 14/20 - Loss: 0.2132
Epoch 15/20 - Loss: 0.2065
Epoch 16/20 - Loss: 0.2171
Epoch 17/20 - Loss: 0.2235
Epoch 18/20 - Loss: 0.3287
Epoch 19/20 - Loss: 0.2130
Epoch 20/20 - Loss: 0.2622


In [9]:
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for data, target in test_loader:
        output = model(data)
        _, preds = torch.max(output, 1)
        all_preds.extend(preds.numpy())
        all_targets.extend(target.numpy())

print("\n--- Evaluation Metrics ---")
print(f"Accuracy:  {accuracy_score(all_targets, all_preds):.4f}")
print(f"Precision: {precision_score(all_targets, all_preds):.4f}")
print(f"Recall:    {recall_score(all_targets, all_preds):.4f}")
print(f"F1 Score:  {f1_score(all_targets, all_preds):.4f}")
print("\nClassification Report:\n", classification_report(all_targets, all_preds))


--- Evaluation Metrics ---
Accuracy:  0.6840
Precision: 0.9635
Recall:    0.4632
F1 Score:  0.6256

Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.98      0.73       215
           1       0.96      0.46      0.63       285

    accuracy                           0.68       500
   macro avg       0.77      0.72      0.68       500
weighted avg       0.80      0.68      0.67       500

